In [2]:
import mlflow
import mlflow.sklearn
import pandas as pd
import numpy as np
from sklearn.decomposition import TruncatedSVD
from sklearn.metrics import mean_squared_error

# 1. Menentukan nama eksperimen di MLflow
mlflow.set_experiment("production-recommendation-system")

# Memuat data training
df_train = pd.read_csv('../data/processed/train.csv')
user_item_matrix = df_train.pivot(index='user_id', columns='item_id', values='rating').fillna(0)

# 2. Memulai sesi pencatatan MLflow
with mlflow.start_run(run_name="svd_baseline_run"):
    
    # Menentukan parameter model
    n_components = 20
    random_state = 42
    
    print(f"Melatih model SVD dengan n_components={n_components}...")
    
    # Melatih model SVD
    svd = TruncatedSVD(n_components=n_components, random_state=random_state)
    user_factors = svd.fit_transform(user_item_matrix)
    item_factors = svd.components_
    predicted_ratings = np.dot(user_factors, item_factors)
    
    # Menghitung metrik sederhana (RMSE pada matriks)
    mse = mean_squared_error(user_item_matrix.values.flatten(), predicted_ratings.flatten())
    rmse = np.sqrt(mse)
    
    # 3. Mencatat Parameter ke MLflow
    mlflow.log_param("n_components", n_components)
    mlflow.log_param("algorithm", "TruncatedSVD")
    
    # 4. Mencatat Metrik ke MLflow
    mlflow.log_metric("rmse", float(rmse))
    
    # 5. Menyimpan model ke MLflow Registry lokal
    mlflow.sklearn.log_model(svd, "svd_model")
    
    print(f"Eksperimen selesai dicatat! RMSE: {rmse:.4f}")

2026/09/05 19:07:33 INFO mlflow.tracking.fluent: Experiment with name 'production-recommendation-system' does not exist. Creating a new experiment.


Melatih model SVD dengan n_components=20...


2026/09/05 19:07:34 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/09/05 19:07:34 WARNING mlflow.sklearn: Model was missing function: predict. Not logging python_function flavor!


Eksperimen selesai dicatat! RMSE: 0.6568
